In [ ]:
'''
"Given current state, predict next 100 actions"
# Future Prediction
input:  [pos_t0, pos_t1, ..., pos_t99]    # Current state sequence
output: [act_t100, act_t101, ..., act_t199] # Next 100 actions
Better uncertainty handling: VAE's probabilistic nature can model the inherent uncertainty in robot actions
Regularized latent space: The KL divergence term prevents overfitting by enforcing a structured latent space
Smooth manifold: Actions become points on a continuous manifold, enabling better interpolation
Improved generalization: Sampling from latent distributions creates more robust predictions
'''
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

if qpos is None:
    print("Failed to load any data!")
    exit()

# Create DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps across all episodes)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("\nDataFrame Summary:")
print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

# Optional: Plot data distribution across episodes
plt.figure(figsize=(15, 10))

joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    plt.subplot(2, 3, i+1)
    plt.plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'QPos {joint}')
    plt.plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Action {joint}')
    plt.title(f'{joint_name} Joint ({joint}) - All Episodes')
    plt.xlabel('Timestep')
    plt.ylabel('Joint Value (radians)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.suptitle('Joint Data Across All Episodes', fontsize=16)
plt.tight_layout()
plt.show()

print("\nData loading complete! Now you can proceed with preprocessing and training.")

In [ ]:
# Complete the normalization code
joint_cols = ['b', 's', 'e', 't', 'r', 'g']

qpos_normalized = {}
action_normalized = {}

for joint in joint_cols:
    # QPos processing
    qpos_joint = qpos_df[joint].fillna(method="ffill")
    qpos_joint = np.array(qpos_joint)
    
    # Simple differences (like velocities)
    qpos_diff = np.diff(qpos_joint)
    qpos_csum_diff = qpos_diff.cumsum()
    
    # Z-score normalization of differences
    qpos_diff_normalized = (qpos_diff - np.mean(qpos_diff)) / (np.std(qpos_diff) + 1e-8)
    
    qpos_normalized[f'{joint}_diff'] = np.concatenate([[0], qpos_diff])
    qpos_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], qpos_csum_diff])
    qpos_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], qpos_diff_normalized])
    
    # Action processing
    action_joint = action_df[joint].fillna(method="ffill")
    action_joint = np.array(action_joint)
    
    action_diff = np.diff(action_joint)
    action_csum_diff = action_diff.cumsum()
    action_diff_normalized = (action_diff - np.mean(action_diff)) / (np.std(action_diff) + 1e-8)
    
    action_normalized[f'{joint}_diff'] = np.concatenate([[0], action_diff])
    action_normalized[f'{joint}_csum_diff'] = np.concatenate([[0], action_csum_diff])
    action_normalized[f'{joint}_diff_norm'] = np.concatenate([[0], action_diff_normalized])

# Create DataFrames from normalized data
qpos_norm_df = pd.DataFrame(qpos_normalized)
qpos_norm_df.insert(0, 'timestamp', qpos_df['timestamp'])

action_norm_df = pd.DataFrame(action_normalized)
action_norm_df.insert(0, 'timestamp', action_df['timestamp'])

print("Normalization completed!")
print(f"QPos normalized shape: {qpos_norm_df.shape}")
print(f"Action normalized shape: {action_norm_df.shape}")



# 2. Model Definition
## 2.1 Positional Encoding Layer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

## 2.2 VAE

In [ ]:
class ActionChunkingVAE(nn.Module):
    def __init__(self, 
                 input_dim=6,        # 6 joint positions
                 output_dim=6,       # 6 joint actions
                 chunk_size=10,      # Predict next 10 actions
                 d_model=256,        # Embedding dimension
                 latent_dim=64,      # Size of latent space
                 nhead=8,            
                 num_layers=6,       
                 dropout=0.1):
        super().__init__()
        
        self.chunk_size = chunk_size
        self.output_dim = output_dim
        self.d_model = d_model
        self.latent_dim = latent_dim
        
        # Input embedding: map joint positions to d_model dimensions
        self.input_embedding = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=False
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # VAE components: map encoder output to latent distribution parameters
        self.mu_encoder = nn.Linear(d_model, latent_dim)
        self.logvar_encoder = nn.Linear(d_model, latent_dim)
        
        # Decoder: map latent vector to action sequence
        self.latent_to_actions = nn.Sequential(
            nn.Linear(latent_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, output_dim * chunk_size)
        )
        
        # Causal mask for encoder
        self.register_buffer('causal_mask', None)
        
        self.init_weights()
    
    def init_weights(self):
        initrange = 0.1
        self.input_embedding.weight.data.uniform_(-initrange, initrange)
        self.mu_encoder.weight.data.uniform_(-initrange, initrange)
        self.logvar_encoder.weight.data.uniform_(-initrange, initrange)
        
        for module in self.latent_to_actions:
            if isinstance(module, nn.Linear):
                module.weight.data.uniform_(-initrange, initrange)
    
    def encode(self, state_sequence):
        """Encode state sequence to latent distribution parameters"""
        seq_len = state_sequence.size(0)
        
        # Generate causal mask if needed
        if self.causal_mask is None or self.causal_mask.size(0) != seq_len:
            mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
            self.causal_mask = mask.masked_fill(mask == 1, float('-inf')).to(state_sequence.device)
        
        # Encode state sequence
        embedded = self.input_embedding(state_sequence)
        with_pos = self.pos_encoder(embedded)
        encoded = self.transformer_encoder(with_pos, self.causal_mask)
        
        # Use only final state representation
        final_state = encoded[-1]  # [batch, d_model]
        
        # Get latent distribution parameters
        # Mean, aka the mu (center of the distribution)
        mu = self.mu_encoder(final_state)          # [batch, latent_dim]
        # log variance, aka the sigma (std of the distribution)
        logvar = self.logvar_encoder(final_state)  # [batch, latent_dim]
        
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """
        Sample from latent distribution using reparameterization trick
        the trick: z = mu + eps * std
        """
        if self.training:
            std = torch.exp(0.5 * logvar)
            # random noise added to the latent space for 
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            # During evaluation, just use the mean
            return mu
    
    def decode(self, z):
        """Decode latent vector to action sequence"""
        batch_size = z.size(0)
        
        # Map latent to flattened action sequence
        actions_flat = self.latent_to_actions(z)  # [batch, output_dim * chunk_size]
        
        # Reshape to action sequence
        action_chunk = actions_flat.view(batch_size, self.chunk_size, self.output_dim)
        
        # Transpose to [chunk_size, batch, output_dim]
        return action_chunk.transpose(0, 1)
    
    def forward(self, state_sequence):
        """Full forward pass"""
        # Encode to latent distribution
        mu, logvar = self.encode(state_sequence)
        
        # Sample from latent distribution
        z = self.reparameterize(mu, logvar)
        
        # Decode to action sequence
        action_chunk = self.decode(z)
        
        # Return predictions and distribution parameters for loss calculation
        return action_chunk, mu, logvar

### Loss function for VAE

In [ ]:
def vae_loss(predicted_actions, target_actions, mu, logvar, kl_weight=0.001):
    """
    VAE loss function: reconstruction loss + KL divergence
    Args:
        predicted_actions: Model predictions [chunk_size, batch, output_dim]
        target_actions: Ground truth [chunk_size, batch, output_dim]
        mu: Mean of latent distribution [batch, latent_dim]
        logvar: Log variance of latent distribution [batch, latent_dim]
        kl_weight: Weight for KL divergence term (beta in beta-VAE)
    """
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(predicted_actions, target_actions)
    
    # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    # Use mean instead of sum for proper normalization
    kl_div = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss (with weighting)
    total_loss = recon_loss + kl_weight * kl_div
    
    return total_loss, recon_loss, kl_div

# 3 Train parameter
## 3.1 Create future window sequence

In [ ]:
# Future prediction
'''
Current State Representation → Optimal Future Action Sequence
# Training pairs
Input:  [states_t0_to_t100]     # 100 timesteps of state history
Target: [actions_t101_to_t110]   # Next 10 actions to execute

# The model learns: "Given this state history, these are the best next actions"
history_len = "state history" = 100
chunk_size = "action chunk" = 10
'''
def create_chunking_sequences(qpos_data, action_data, history_len, chunk_size):
    sequences = []
    
    for i in range(len(qpos_data) - history_len - chunk_size):
        state_history = qpos_data[i:i+history_len]                    # [t0, t1, t2, t3...t99]
        future_actions = action_data[i+history_len:i+history_len+chunk_size]  # [t100, t101, t102...t109]
        # Model learns: state_history → future_actions
        sequences.append((state_history, future_actions))
    return sequences

In [ ]:
def get_action_chunking_data(qpos_df, action_df, history_len=10, chunk_size=10, train_split=0.8):
    """
    Prepare robot data for action chunking training
    Args:
        qpos_df: DataFrame with joint positions
        action_df: DataFrame with joint actions
        history_len: length of state history sequence
        chunk_size: number of future actions to predict
        train_split: fraction of data for training
    Returns:
        train_data: list of (state_history, future_actions) tuples for training
        test_data: list of (state_history, future_actions) tuples for testing
        normalization_params: parameters for denormalizing predictions
    """
    # Extract joint columns (skip timestamp)
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    qpos_data = qpos_df[joint_cols].values  # [timesteps, 6 joints], timesteps should be history_len = 100
    action_data = action_df[joint_cols].values  # [timesteps, 6 joints] , timesteps should be chunk_size = 10

    # Normalize data (important for stable training)
    # Min-max normalization
    qpos_min, qpos_max = qpos_data.min(axis=0), qpos_data.max(axis=0)
    qpos_normalized = (qpos_data - qpos_min) / (qpos_max - qpos_min + 1e-8)
    
    action_min, action_max = action_data.min(axis=0), action_data.max(axis=0)
    action_normalized = (action_data - action_min) / (action_max - action_min + 1e-8)
    
    # Create action chunking sequences
    sequences = create_chunking_sequences(qpos_normalized, action_normalized, history_len, chunk_size)
    
    # Split into train/test
    split_idx = int(len(sequences) * train_split)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    # Convert to tensors
    train_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                  for seq in train_sequences]
    test_data = [(torch.FloatTensor(seq[0]), torch.FloatTensor(seq[1])) 
                 for seq in test_sequences]
    
    # Store normalization parameters for later use
    normalization_params = {
        'qpos_min': qpos_min, 'qpos_max': qpos_max,
        'action_min': action_min, 'action_max': action_max
    }
    
    print(f"Action Chunking Data Preparation:")
    print(f"State history length: {history_len}")
    print(f"Action chunk size: {chunk_size}")
    print(f"Total sequences: {len(sequences)}")
    print(f"Training sequences: {len(train_data)}")
    print(f"Testing sequences: {len(test_data)}")
    print(f"Input shape per sequence: [history_len={history_len}, joints=6]")
    print(f"Output shape per sequence: [chunk_size={chunk_size}, joints=6]")
    
    return train_data, test_data, normalization_params

def get_action_chunking_batch(sequences, start_idx, batch_size):
    """
    Create batches for action chunking data
    Args:
        sequences: list of (state_history, future_actions) tuples
        start_idx: starting index
        batch_size: batch size
    Returns:
        state_histories: [history_len, batch, 6] - batched state sequences
        future_actions: [chunk_size, batch, 6] - batched future action chunks
    """
    end_idx = min(start_idx + batch_size, len(sequences))
    batch_sequences = sequences[start_idx:end_idx]
    
    # Stack sequences into batches
    state_histories = torch.stack([seq[0] for seq in batch_sequences], dim=1)  # [history_len, batch, 6]
    future_actions = torch.stack([seq[1] for seq in batch_sequences], dim=1)   # [chunk_size, batch, 6]
    
    return state_histories, future_actions

# Usage example:
history_len = 100  # Use 100 timesteps of state history
chunk_size = 10   # Predict next 10 actions

train_data, test_data, norm_params = get_action_chunking_data(
    qpos_df, action_df, 
    history_len=history_len,
    chunk_size=chunk_size,
    train_split=0.8
)

# Set model parameter

In [ ]:
# Initialize transformer model with 256-dimensional embeddings, 8 heads for 
# Model parameters (make sure these match your data preparation)
model = ActionChunkingVAE(
    input_dim=6,        # 6 joint positions
    output_dim=6,       # 6 joint actions  
    chunk_size=10,      # Predict next 10 actions
    d_model=256,        # Embedding dimension
    latent_dim=256,      # Size of latent space
    nhead=8,            # Attention heads
    num_layers=6,       # Transformer layers
    dropout=0.1
).to(device)


lr = 0.00005
epochs = 2000
batch_size = 32
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

## Training function

In [ ]:
def train_vae_action_chunking(model, train_data, val_data, device, epochs=200, batch_size=32, kl_weight=0.001):
    """Train VAE action chunking model"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)
    
    train_losses = []
    val_losses = []
    recon_losses = []
    kl_losses = []
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0.0
        total_recon_loss = 0.0
        total_kl_loss = 0.0
        num_batches = 0
        
        for i in range(0, len(train_data), batch_size):
            # Get batch
            state_histories, future_actions = get_action_chunking_batch(train_data, i, batch_size)
            state_histories = state_histories.to(device)
            future_actions = future_actions.to(device)
            
            # Forward pass
            predicted_actions, mu, logvar = model(state_histories)
            
            # Calculate loss
            loss, recon_loss, kl_loss = vae_loss(
                predicted_actions, future_actions, mu, logvar, kl_weight
            )
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Track losses
            total_train_loss += loss.item()
            total_recon_loss += recon_loss.item()
            total_kl_loss += kl_loss.item()
            num_batches += 1
        
        # Calculate average losses
        avg_train_loss = total_train_loss / num_batches
        avg_recon_loss = total_recon_loss / num_batches
        avg_kl_loss = total_kl_loss / num_batches
        
        train_losses.append(avg_train_loss)
        recon_losses.append(avg_recon_loss)
        kl_losses.append(avg_kl_loss)
        
        # Validation phase
        model.eval()
        total_val_loss = 0.0
        total_val_recon = 0.0
        total_val_kl = 0.0
        num_val_batches = 0
        
        with torch.no_grad():
            for i in range(0, len(val_data), batch_size):
                state_histories, future_actions = get_action_chunking_batch(val_data, i, batch_size)
                state_histories, future_actions = state_histories.to(device), future_actions.to(device)
                
                # ✅ Correct: unpack VAE output
                predicted_actions, mu, logvar = model(state_histories)
                loss, recon_loss, kl_loss = vae_loss(
                    predicted_actions, future_actions, mu, logvar, kl_weight
                )
                
                total_val_loss += loss.item()
                total_val_recon += recon_loss.item()
                total_val_kl += kl_loss.item()
                num_val_batches += 1
        
        avg_val_loss = total_val_loss / num_val_batches if num_val_batches > 0 else float('inf')
        avg_val_recon = total_val_recon / num_val_batches if num_val_batches > 0 else float('inf')
        avg_val_kl = total_val_kl / num_val_batches if num_val_batches > 0 else float('inf')
        
        val_losses.append(avg_val_loss)
        
        # Print epoch summary
        print(f"Epoch {epoch:3d} | Train Loss: {avg_train_loss:.6f} | "
              f"Recon: {avg_recon_loss:.6f} | KL: {avg_kl_loss:.6f} | "
              f"Val Loss: {avg_val_loss:.6f} | Val Recon: {avg_val_recon:.6f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        scheduler.step(avg_val_loss)
        
    return train_losses, val_losses, recon_losses, kl_losses

# Train the model

In [ ]:
print(f"Using device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train the VAE model
kl_weight = 0.001  # Start with small KL weight
train_losses, val_losses, recon_losses, kl_losses = train_vae_action_chunking(
    model, train_data, test_data, device, epochs=epochs, batch_size=batch_size, kl_weight=kl_weight
)

# Plot training progress including VAE-specific losses
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('VAE Total Loss')
plt.legend()
plt.yscale('log')

plt.subplot(2, 2, 2)
plt.plot(recon_losses, label='Reconstruction Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Reconstruction Loss')
plt.legend()
plt.yscale('log')

plt.subplot(2, 2, 3)
plt.plot(kl_losses, label='KL Divergence')
plt.xlabel('Epoch')
plt.ylabel('KL Loss')
plt.title('KL Divergence')
plt.legend()

plt.subplot(2, 2, 4)
plt.plot(train_losses[-50:], label='Train Loss (Last 50)')
plt.plot(val_losses[-50:], label='Val Loss (Last 50)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Recent Training Progress')
plt.legend()

plt.tight_layout()
plt.show()

## Evaluation function

In [ ]:
def evaluate_action_chunking_model(model, val_data, norm_params, device, num_sequences=5):
    """
    Evaluate action chunking model with future prediction visualization
    Args:
        model: trained action chunking transformer model
        val_data: validation sequences (state_history, future_actions) tuples
        norm_params: normalization parameters for denormalizing predictions
        device: torch device
        num_sequences: number of validation sequences to evaluate
    Returns:
        avg_loss: average validation loss
        predictions: denormalized predictions
        actuals: denormalized actual values
    """
    model.eval()
    criterion = nn.MSELoss()
    
    joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Storage for results
    all_predictions = []
    all_actuals = []
    all_losses = []
    
    with torch.no_grad():
        # Select random sequences for detailed evaluation
        selected_indices = np.random.choice(len(val_data), 
                                          size=min(num_sequences, len(val_data)), 
                                          replace=False)
        
        print(f"Evaluating {len(selected_indices)} sequences")
        print(f"Each sequence: state_history → future_actions (chunk_size={model.chunk_size})")
        
        for seq_idx in selected_indices:
            # Get one sequence: (state_history, future_actions)
            state_history, future_actions = val_data[seq_idx]
            
            # Prepare input: add batch dimension
            state_input = state_history.unsqueeze(1).to(device)  # [history_len, 1, 6]
            future_target = future_actions.to(device)  # [chunk_size, 6]
            
            # Predict future actions from state history
            predicted_chunk, _, _ = model(state_input)  # Get only predictions, ignore mu, logvar [chunk_size, 1, 6]
            predicted_chunk = predicted_chunk.squeeze(1)  # [chunk_size, 6]
            
            # Store predictions and actuals
            all_predictions.append(predicted_chunk.cpu())  # [chunk_size, 6]
            all_actuals.append(future_target.cpu())        # [chunk_size, 6]
            
            # Calculate loss for this sequence
            seq_loss = criterion(predicted_chunk, future_target).item()
            all_losses.append(seq_loss)
    
    # Calculate average loss
    avg_loss = np.mean(all_losses) if all_losses else float('inf')
    
    # Denormalize predictions and actuals
    if 'action_min' in norm_params and 'action_max' in norm_params:
        # Min-max denormalization: value = normalized * (max - min) + min
        action_min = torch.FloatTensor(norm_params['action_min'])
        action_max = torch.FloatTensor(norm_params['action_max'])
        action_range = action_max - action_min
        
        denorm_predictions = []
        denorm_actuals = []
        
        for pred, actual in zip(all_predictions, all_actuals):
            # Denormalize: value = normalized * (max - min) + min
            denorm_pred = pred * action_range + action_min
            denorm_actual = actual * action_range + action_min
            
            # Convert to numpy for plotting
            denorm_predictions.append(denorm_pred.numpy())
            denorm_actuals.append(denorm_actual.numpy())
    
    elif 'action_mean' in norm_params and 'action_std' in norm_params:
        # Z-score denormalization: value = normalized * std + mean
        action_mean = torch.FloatTensor(norm_params['action_mean'])
        action_std = torch.FloatTensor(norm_params['action_std'])
        
        denorm_predictions = []
        denorm_actuals = []
        
        for pred, actual in zip(all_predictions, all_actuals):
            # Denormalize: value = normalized * std + mean
            denorm_pred = pred * action_std + action_mean
            denorm_actual = actual * action_std + action_mean
            
            # Convert to numpy for plotting
            denorm_predictions.append(denorm_pred.numpy())
            denorm_actuals.append(denorm_actual.numpy())
    
    else:
        # Fallback: use normalized values if no proper normalization params found
        print("Warning: No proper normalization parameters found. Using normalized values.")
        denorm_predictions = [pred.numpy() for pred in all_predictions]
        denorm_actuals = [actual.numpy() for actual in all_actuals]
    
    return avg_loss, denorm_predictions, denorm_actuals, joint_names, joint_cols


def plot_action_chunking_results(predictions, actuals, joint_names, joint_cols, chunk_size=None):
    """
    Create comprehensive plots comparing predicted vs actual future actions
    """
    num_sequences = len(predictions)
    
    # Auto-detect chunk size if not provided
    if chunk_size is None:
        chunk_size = len(predictions[0]) if predictions else 10
    
    # Plot 1: All joints for first sequence (future action prediction)
    if num_sequences > 0:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        pred_seq = predictions[0]  # [chunk_size, 6] - numpy array
        actual_seq = actuals[0]    # [chunk_size, 6] - numpy array
        
        for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
            future_steps = range(chunk_size)
            
            axes[i].plot(future_steps, actual_seq[:chunk_size, i], 'b-o', linewidth=2, 
                        label=f'Actual Future {joint_col}', alpha=0.8)
            axes[i].plot(future_steps, pred_seq[:chunk_size, i], 'r--s', linewidth=2, 
                        label=f'Predicted Future {joint_col}', alpha=0.8)
            
            axes[i].set_title(f'{joint_name} Joint ({joint_col}) - Future Actions')
            axes[i].set_xlabel('Future Time Step')
            axes[i].set_ylabel('Joint Action Value')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
            
            # Calculate and display error metrics
            mse = np.mean((actual_seq[:chunk_size, i] - pred_seq[:chunk_size, i])**2)
            mae = np.mean(np.abs(actual_seq[:chunk_size, i] - pred_seq[:chunk_size, i]))
            axes[i].text(0.05, 0.95, f'MSE: {mse:.4f}\nMAE: {mae:.4f}', 
                        transform=axes[i].transAxes, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.suptitle(f'Future Action Predictions vs Actuals - Sequence 1 ({chunk_size} future steps)', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Plot 2: Error analysis across all sequences
    if num_sequences > 1:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
            errors_per_step = []
            
            for step in range(chunk_size):
                step_errors = []
                for seq_idx in range(num_sequences):
                    if step < len(predictions[seq_idx]) and step < len(actuals[seq_idx]):
                        error = abs(actuals[seq_idx][step, i] - predictions[seq_idx][step, i])
                        step_errors.append(error)
                
                if step_errors:
                    errors_per_step.append(step_errors)
            
            # Box plot of errors across sequences for each future time step
            if errors_per_step:
                axes[i].boxplot(errors_per_step, positions=range(1, len(errors_per_step)+1))
                axes[i].set_title(f'{joint_name} Joint ({joint_col}) - Future Prediction Error Distribution')
                axes[i].set_xlabel('Future Time Step')
                axes[i].set_ylabel('Absolute Error')
                axes[i].grid(True, alpha=0.3)
        
        plt.suptitle(f'Future Prediction Error Analysis Across {num_sequences} Sequences', fontsize=16)
        plt.tight_layout()
        plt.show()
    
    # Plot 3: Correlation analysis for future predictions
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, (joint_name, joint_col) in enumerate(zip(joint_names, joint_cols)):
        all_actual_vals = []
        all_pred_vals = []
        
        for seq_idx in range(num_sequences):
            # Use all available future steps for correlation analysis
            seq_length = min(len(predictions[seq_idx]), len(actuals[seq_idx]))
            all_actual_vals.extend(actuals[seq_idx][:seq_length, i].tolist())
            all_pred_vals.extend(predictions[seq_idx][:seq_length, i].tolist())
        
        if all_actual_vals and all_pred_vals:
            axes[i].scatter(all_actual_vals, all_pred_vals, alpha=0.6, s=20)
            
            # Perfect prediction line
            min_val = min(min(all_actual_vals), min(all_pred_vals))
            max_val = max(max(all_actual_vals), max(all_pred_vals))
            axes[i].plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8, label='Perfect Prediction')
            
            # Calculate correlation
            correlation = np.corrcoef(all_actual_vals, all_pred_vals)[0, 1]
            
            axes[i].set_title(f'{joint_name} Joint ({joint_col})\nCorr: {correlation:.3f}')
            axes[i].set_xlabel('Actual Future Actions')
            axes[i].set_ylabel('Predicted Future Actions')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
    
    plt.suptitle('Predicted vs Actual Future Actions Correlation Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()


def comprehensive_action_chunking_evaluation(model, val_data, norm_params, device):
    """
    Complete evaluation pipeline for action chunking
    """
    print("Starting comprehensive action chunking model evaluation...")
    print("="*60)
    
    # Evaluate model
    avg_loss, predictions, actuals, joint_names, joint_cols = evaluate_action_chunking_model(
        model, val_data, norm_params, device, num_sequences=5
    )
    
    print(f"Average Validation Loss: {avg_loss:.6f}")
    
    # Calculate overall statistics
    if predictions and actuals:
        all_errors = []
        joint_errors = {joint: [] for joint in joint_cols}
        
        for pred_seq, actual_seq in zip(predictions, actuals):
            seq_length = min(len(pred_seq), len(actual_seq))
            for step in range(seq_length):
                for joint_idx, joint in enumerate(joint_cols):
                    error = abs(actual_seq[step, joint_idx] - pred_seq[step, joint_idx])
                    all_errors.append(error)
                    joint_errors[joint].append(error)
        
        print(f"\nOverall Statistics:")
        print(f"Mean Absolute Error: {np.mean(all_errors):.4f}")
        print(f"Max Absolute Error: {np.max(all_errors):.4f}")
        print(f"Std Absolute Error: {np.std(all_errors):.4f}")
        
        print(f"\nPer-Joint Statistics (Future Actions):")
        for joint in joint_cols:
            if joint_errors[joint]:
                print(f"Joint {joint}: MAE = {np.mean(joint_errors[joint]):.4f}, "
                      f"Max = {np.max(joint_errors[joint]):.4f}")
    
    # Create plots
    if predictions and actuals:
        plot_action_chunking_results(predictions, actuals, joint_names, joint_cols, model.chunk_size)
    
    return avg_loss, predictions, actuals

# Usage after training your action chunking model:
avg_loss, predictions, actuals = comprehensive_action_chunking_evaluation(
    model, test_data, norm_params, device
)